<a href="https://colab.research.google.com/github/jabin-hao/reserach-LTFormer/blob/main/LTFormer.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# LTFormer Colab 复现指南（极速多进程 + 断点续传版）

> **LTFormer**: *A light-weight transformer-based self-supervised matching network for heterogeneous remote sensing images* (*Information Fusion*, 2024)

### 使用流程建议
1. **阶段一（CPU High-RAM 运行时）**：运行 **步骤 1 ~ 步骤 4**，利用多核 CPU + Colab 本地 SSD 极速将 `whu-opt-sar` 原始大图切分并生成自监督三元组，按分片（`chunk_XXXX.tar`）实时保存到 Google Drive。**如中途断开，重连后依次重跑步骤 1~4 即可自动跳过已完成的分片断点续传。**
2. **阶段二（GPU 运行时）**：三元组全部生成完成后，在 Colab 菜单选择 `代码执行程序 -> 更改运行时类型 -> GPU`，依次运行 **步骤 1、步骤 2、步骤 5、步骤 6**，将分片快速解压到本地 SSD 并开始训练，权重实时保存回 Google Drive。
3. **阶段三（GPU 运行时 - 评估与官方权重对照）**：训练完成后，在 GPU 运行时下（重连时先跑 **步骤 1、步骤 2**）依次运行 **步骤 7 ~ 步骤 9**，自动构造符合 `PatchDataset` 规范的测试集，并同口径对比 **你的训练权重** vs **官方 `LTFormer.pth`** vs **官方 `LTFormer_hardnet_loss.pth`** 的 `Precision` 与 `Matching_score`，同时在 Notebook 中直接可视化匹配连线图。

### 步骤 1: 挂载 Google Drive 并检查当前硬件环境

In [1]:
from google.colab import drive
import os, psutil, torch

drive.mount('/content/drive')

print(f"CPU 核心数: {os.cpu_count()}")
print(f"系统总内存: {psutil.virtual_memory().total / (1024**3):.1f} GB")
print(f"GPU 可用: {torch.cuda.is_available()}", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "")

Mounted at /content/drive
CPU 核心数: 8
系统总内存: 51.0 GB
GPU 可用: True Tesla T4


### 步骤 2: 克隆 / 更新 Fork 仓库并安装运行依赖

In [5]:
import os
%cd /content
if not os.path.exists('/content/article-LTFormer'):
    !git clone https://github.com/jabin-hao/article-LTFormer.git
else:
    %cd /content/article-LTFormer
    !git pull

%cd /content/article-LTFormer
!pip install -q hydra-core omegaconf timm tifffile pyyaml tqdm imageio scipy imagecodecs munkres
!ls -la

/content
Cloning into 'article-LTFormer'...
remote: Enumerating objects: 153, done.
remote: Counting objects: 100% (79/79), done.
remote: Compressing objects: 100% (48/48), done.
remote: Total 153 (delta 35), reused 60 (delta 31), pack-reused 74 (from 1)
Receiving objects: 100% (153/153), 18.38 MiB | 14.00 MiB/s, done.
Resolving deltas: 100% (55/55), done.
/content/article-LTFormer
total 116
drwxr-xr-x 7 root root  4096 Nov 25 03:31 .
drwxr-xr-x 1 root root  4096 Nov 25 03:31 ..
drwxr-xr-x 2 root root  4096 Nov 25 03:31 config
-rw-r--r-- 1 root root  4669 Nov 25 03:31 environment.yml
drwxr-xr-x 8 root root  4096 Nov 25 03:31 .git
-rw-r--r-- 1 root root 12356 Nov 25 03:31 LTFormer.ipynb
drwxr-xr-x 2 root root  4096 Nov 25 03:31 models
drwxr-xr-x 2 root root  4096 Nov 25 03:31 pic
-rw-r--r-- 1 root root  2544 Nov 25 03:31 README.md
-rw-r--r-- 1 root root 16706 Nov 25 03:31 run_generate_dataset_fast.py
-rw-r--r-- 1 root root 10246 Nov 25 03:31 run_generate_dataset.py
-rw-r--r-- 1 root roo

### 步骤 3: 配置 Google Drive 工作区路径并探查原始大图规格

In [6]:
import os, glob
from run_generate_dataset_fast import collect_images, read_image_any

WORKSPACE_DIR = "/content/drive/MyDrive/paper/LTFormer_workspace"
RAW_OPTICAL_DIR = os.path.join(WORKSPACE_DIR, "data/whu-opt-sar/optical")
RAW_SAR_DIR = os.path.join(WORKSPACE_DIR, "data/whu-opt-sar/sar")
DRIVE_CHUNKS_DIR = os.path.join(WORKSPACE_DIR, "triplets_chunks")
DRIVE_TRAIN_CKPT_DIR = os.path.join(WORKSPACE_DIR, "train_outputs")

os.makedirs(DRIVE_CHUNKS_DIR, exist_ok=True)
os.makedirs(DRIVE_TRAIN_CKPT_DIR, exist_ok=True)

opt_files = collect_images(RAW_OPTICAL_DIR)
sar_files = collect_images(RAW_SAR_DIR)
print(f"optical 图像数: {len(opt_files)}")
print(f"sar     图像数: {len(sar_files)}")

if opt_files:
    sample_opt = read_image_any(opt_files[0])
    print(f"示例 optical 文件: {os.path.basename(opt_files[0])}, shape={sample_opt.shape}, dtype={sample_opt.dtype}")
if sar_files:
    sample_sar = read_image_any(sar_files[0])
    print(f"示例 sar     文件: {os.path.basename(sar_files[0])}, shape={sample_sar.shape}, dtype={sample_sar.dtype}")

optical 图像数: 100
sar     图像数: 100
示例 optical 文件: NH49E001013.tif, shape=(3704, 5556, 4), dtype=uint8
示例 sar     文件: NH49E001013.tif, shape=(3704, 5556), dtype=uint8


### 步骤 4（CPU 运行时）: 多核极速 + 断点续传生成三元组
- `--mode auto`：若 `optical` 为 4 波段（RGB+NIR），按论文 README 自动剥离可见光与近红外波段并切分成 `926×926`；若为 3 波段，则自动与 `sar` 配对并切分成 `926×926`。
- 源图先并发缓存到 Colab 本地 SSD `/content/ltformer_cache`，处理完每个分片后立即打包为 `chunk_XXXX.tar` 保存到 Google Drive。
- 若中途断连，重新运行本单元格会自动跳过 Drive 中已生成的 `chunk_*.tar`。

In [ ]:
import os, glob

!python run_generate_dataset_fast.py \
    --rgb-dir "{RAW_OPTICAL_DIR}" \
    --pair-dir "{RAW_SAR_DIR}" \
    --out-dir "{DRIVE_CHUNKS_DIR}" \
    --cache-dir "/content/ltformer_cache" \
    --mode auto \
    --tile-size 926 \
    --patch-half 128 \
    --chunk-size 5

chunks = sorted(glob.glob(os.path.join(DRIVE_CHUNKS_DIR, "chunk_*.tar")))
total_mb = sum(os.path.getsize(c) for c in chunks) / (1024 ** 2)
print(f"\n✅ Google Drive 当前共有 {len(chunks)} 个三元组分片，总计 {total_mb:.1f} MB")

### 步骤 5（切换 GPU 运行时后执行）: 从 Google Drive 解压三元组到本地 SSD
切到 GPU 运行时后先执行 **步骤 1 ~ 步骤 3**，再执行本单元格。把分片解压到 `/content/triplets_train`，避免训练时直接读写 Drive 导致 GPU 饿死。

In [6]:
import os, glob, tarfile, torch
from concurrent.futures import ThreadPoolExecutor
from tqdm.auto import tqdm

assert torch.cuda.is_available(), "⚠️ 当前未检测到 GPU！请先切换 Colab 运行时为 GPU 后再继续。"

if 'DRIVE_CHUNKS_DIR' not in globals():
    DRIVE_CHUNKS_DIR = "/content/drive/MyDrive/paper/LTFormer_workspace/triplets_chunks"
if 'DRIVE_TRAIN_CKPT_DIR' not in globals():
    DRIVE_TRAIN_CKPT_DIR = "/content/drive/MyDrive/paper/LTFormer_workspace/train_outputs"
os.makedirs(DRIVE_TRAIN_CKPT_DIR, exist_ok=True)

LOCAL_TRAIN_DIR = "/content/triplets_train"
os.makedirs(LOCAL_TRAIN_DIR, exist_ok=True)

chunk_tars = sorted(glob.glob(os.path.join(DRIVE_CHUNKS_DIR, "chunk_*.tar")))
assert len(chunk_tars) > 0, f"未在 {DRIVE_CHUNKS_DIR} 找到 chunk_*.tar，请先完成步骤 4！"

def extract_one(tar_path):
    with tarfile.open(tar_path, "r") as tar:
        tar.extractall(LOCAL_TRAIN_DIR)

with ThreadPoolExecutor(max_workers=4) as ex:
    list(tqdm(ex.map(extract_one, chunk_tars), total=len(chunk_tars), desc="解压三元组到本地SSD"))

num_folders = len(os.listdir(LOCAL_TRAIN_DIR))
print(f"🎉 解压完成！本地可用于训练的三元组样本文件夹数: {num_folders}")

解压三元组到本地SSD:   0%|          | 0/20 [00:00<?, ?it/s]

/tmp/ipykernel_486/3724995241.py:21: DeprecationWarning: Python 3.14 will, by default, filter extracted tar archives and reject files or modify their metadata. Use the filter argument to control this behavior.
  tar.extractall(LOCAL_TRAIN_DIR)


🎉 解压完成！本地可用于训练的三元组样本文件夹数: 184281


### 步骤 6（GPU 运行时）: 启动自监督训练（Checkpoint 自动保存至 Google Drive）

In [7]:
!python run_training_ltformer.py \
    paths.train_data="/content/triplets_train" \
    hydra.run.dir="{DRIVE_TRAIN_CKPT_DIR}"

print(f"✅ 训练结束，模型权重已保存在: {DRIVE_TRAIN_CKPT_DIR}")

CUDA is available. Training on GPU
I0000 00:00:1795506517.139006    4516 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
I0000 00:00:1795506521.474845    4516 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
/usr/local/lib/python3.13/dist-packages/timm/models/layers/__init__.py:49: FutureWarning: Importing from timm.models.layers is deprecated, please import via timm.layers
  warnings.warn(f"Importing from {__name__} is deprecated, please import via timm.layers", FutureWarning)
/usr/local/lib/python3.13/dist-packages/timm/models/registry.py:4: FutureWarning: Importing from timm.models.registry is deprecated, please import via timm.models
  warnings.warn(f"Importing from {__name__} is deprecated, please import via timm.models", FutureWar

### 步骤 7（GPU 运行时）: 检查权重并构建跨模态 + 仿射几何扰动测试集
- 自动定位你的训练权重（`train_outputs/LTFormer.pth`）及官方权重（`models/LTFormer.pth`、`models/LTFormer_hardnet_loss.pth`）。
- 若 Drive 中尚未准备 `data/test_cut`，则从 `whu-opt-sar` 抽取 20 组 `926×926` **真实异源图像对**（RGB vs NIR 第 4 波段，或 RGB vs SAR），并对目标图施加**旋转（±8°~15°）+ 缩放（0.9~1.1）+ 平移（±8px）**仿射变换，提取高密度 SIFT 关键点（`nfeatures=250`）写入 `/content/test_cut_stress`，用于真实拉开不同模型的性能差距。

In [ ]:
import os, glob, cv2, torch
import numpy as np
from run_generate_dataset_fast import collect_images, read_image_any, to_uint8

assert torch.cuda.is_available(), "⚠️ 当前未检测到 GPU！请先切换 Colab 运行时为 GPU 后再继续（run_matching_demo.py 依赖 CUDA）。"

WORKSPACE_DIR = "/content/drive/MyDrive/paper/LTFormer_workspace"
DRIVE_TRAIN_CKPT_DIR = os.path.join(WORKSPACE_DIR, "train_outputs")
DRIVE_EVAL_DIR = os.path.join(WORKSPACE_DIR, "eval_outputs")
os.makedirs(DRIVE_EVAL_DIR, exist_ok=True)

MY_WEIGHTS = os.path.join(DRIVE_TRAIN_CKPT_DIR, "LTFormer.pth")
OFFICIAL_WEIGHTS = "/content/article-LTFormer/models/LTFormer.pth"
HARDNET_LOSS_WEIGHTS = "/content/article-LTFormer/models/LTFormer_hardnet_loss.pth"

for name, p in [("你的训练权重", MY_WEIGHTS), ("官方 LTFormer", OFFICIAL_WEIGHTS), ("官方 HardNetLoss", HARDNET_LOSS_WEIGHTS)]:
    status = "✅ 存在" if os.path.exists(p) else "❌ 未找到"
    print(f"{name:16s}: {status} -> {p}")

DRIVE_TEST_DIR = os.path.join(WORKSPACE_DIR, "data/test_cut")
if os.path.exists(DRIVE_TEST_DIR) and len(glob.glob(os.path.join(DRIVE_TEST_DIR, "*", "matches", "*", "*.txt"))) > 0:
    TEST_DATA_DIR = DRIVE_TEST_DIR
    print(f"\n✅ 使用 Drive 已有测试集: {TEST_DATA_DIR}")
else:
    TEST_DATA_DIR = "/content/test_cut_stress"
    seq_dir = os.path.join(TEST_DATA_DIR, "seq_01")
    if not os.path.exists(seq_dir) or len(glob.glob(os.path.join(seq_dir, "matches", "*", "*.txt"))) == 0:
        print(f"\n⚙️ 正在构建带仿射几何扰动的跨模态压力测试集 -> {TEST_DATA_DIR} ...")
        raw_opt_dir = os.path.join(WORKSPACE_DIR, "data/whu-opt-sar/optical")
        raw_sar_dir = os.path.join(WORKSPACE_DIR, "data/whu-opt-sar/sar")
        opt_files = sorted(collect_images(raw_opt_dir))
        sar_files = sorted(collect_images(raw_sar_dir))
        sar_map = {os.path.splitext(os.path.basename(p))[0]: p for p in sar_files}
        assert len(opt_files) > 0, f"未找到原始图像: {raw_opt_dir}"

        rgb_out = os.path.join(seq_dir, "rgb", "sample")
        nir_out = os.path.join(seq_dir, "NIRS", "sample")
        mat_out = os.path.join(seq_dir, "matches", "sample")
        os.makedirs(rgb_out, exist_ok=True)
        os.makedirs(nir_out, exist_ok=True)
        os.makedirs(mat_out, exist_ok=True)

        sift = cv2.SIFT_create(nfeatures=250)
        angles = [10, -12, 15, -10, 8]
        scales = [0.90, 1.10, 0.92, 1.08, 0.95]
        shifts = [(8, -6), (-8, 8), (6, 8), (-6, -8), (10, 5)]

        saved = 0
        pair_mode = None
        start_idx = max(0, len(opt_files) - 20)
        for idx, fp in enumerate(opt_files[start_idx:]):
            img = read_image_any(fp)
            h, w = img.shape[:2]
            y0, x0 = max(0, (h - 926) // 2), max(0, (w - 926) // 2)
            crop = img[y0:y0 + 926, x0:x0 + 926]
            if crop.ndim == 3 and crop.shape[2] >= 4:
                rgb_img = to_uint8(crop[:, :, :3])
                nir_img = cv2.cvtColor(to_uint8(crop[:, :, 3]), cv2.COLOR_GRAY2BGR)
                pair_mode = "RGB vs NIR (4波段分离)"
            else:
                rgb_img = to_uint8(crop[:, :, :3]) if crop.ndim == 3 else cv2.cvtColor(to_uint8(crop), cv2.COLOR_GRAY2BGR)
                stem = os.path.splitext(os.path.basename(fp))[0]
                sar_fp = sar_map.get(stem) or (sar_files[start_idx + idx] if (start_idx + idx) < len(sar_files) else None)
                assert sar_fp is not None and os.path.exists(sar_fp), f"未找到与 {fp} 配对的 SAR 图像！"
                sar_crop = read_image_any(sar_fp)[y0:y0 + 926, x0:x0 + 926]
                sar_u8 = to_uint8(sar_crop)
                nir_img = sar_u8[:, :, :3] if sar_u8.ndim == 3 else cv2.cvtColor(sar_u8, cv2.COLOR_GRAY2BGR)
                pair_mode = "RGB vs SAR"

            th, tw = nir_img.shape[:2]
            angle = angles[idx % len(angles)]
            scale = scales[idx % len(scales)]
            tx, ty = shifts[idx % len(shifts)]
            M = cv2.getRotationMatrix2D((tw / 2.0, th / 2.0), angle, scale)
            M[0, 2] += tx
            M[1, 2] += ty
            nir_warped = cv2.warpAffine(nir_img, M, (tw, th), flags=cv2.INTER_LINEAR, borderMode=cv2.BORDER_REFLECT_101)

            gray = cv2.cvtColor(rgb_img, cv2.COLOR_BGR2GRAY)
            kps = sift.detect(gray, None)
            pairs = []
            for kp in kps:
                xs, ys = kp.pt[0], kp.pt[1]
                xd = M[0, 0] * xs + M[0, 1] * ys + M[0, 2]
                yd = M[1, 0] * xs + M[1, 1] * ys + M[1, 2]
                if 70 < xs < tw - 70 and 70 < ys < th - 70 and 70 < xd < tw - 70 and 70 < yd < th - 70:
                    pairs.append((xs, ys, xd, yd))
            if len(pairs) < 20:
                continue
            stem = os.path.splitext(os.path.basename(fp))[0]
            cv2.imwrite(os.path.join(rgb_out, f"{stem}.png"), rgb_img)
            cv2.imwrite(os.path.join(nir_out, f"{stem}.png"), nir_warped)
            with open(os.path.join(mat_out, f"{stem}.txt"), "w") as f:
                for xs, ys, xd, yd in pairs:
                    f.write(f"{xs:.2f} {ys:.2f} {xd:.2f} {yd:.2f}\n")
            saved += 1
        print(f"✅ 压力测试集构建完成（模式: {pair_mode}），共 {saved} 对图像 -> {TEST_DATA_DIR}")
    else:
        print(f"✅ 复用本地已构建的压力测试集: {TEST_DATA_DIR}")

### 步骤 8（GPU 运行时）: 运行定量评估与同口径对照（`Precision` & `Matching_score`）
在同一份带几何扰动的跨模态测试集（`patch_size=64`、`image_size=128`、`matching_threshold=0.35`）下，依次评估：
1. **你的 Colab 训练权重**（`train_outputs/LTFormer.pth`）
2. **官方 LTFormer 权重**（`models/LTFormer.pth`，对应论文 `LT Loss`）
3. **官方基线权重**（`models/LTFormer_hardnet_loss.pth`，对应论文 Table 6 `HardNet Loss`）

In [ ]:
import os, re, subprocess

eval_targets = [
    ("My_LTFormer (Colab)", MY_WEIGHTS, os.path.join(DRIVE_EVAL_DIR, "my_ltformer")),
    ("Official_LTFormer (LT Loss)", OFFICIAL_WEIGHTS, os.path.join(DRIVE_EVAL_DIR, "official_ltformer")),
    ("Official_HardNetLoss (Baseline)", HARDNET_LOSS_WEIGHTS, os.path.join(DRIVE_EVAL_DIR, "official_hardnet_loss")),
]

results = []
for label, w_path, out_dir in eval_targets:
    if not os.path.exists(w_path):
        print(f"⏭️ 跳过 {label}，权重不存在: {w_path}")
        continue
    os.makedirs(out_dir, exist_ok=True)
    print(f"\n==================== 正在评估: {label} ====================")
    cmd = [
        "python", "run_matching_demo.py",
        f"paths.demo_sequence_data={TEST_DATA_DIR}",
        f"params.weights_path={w_path}",
        "params.matching_threshold=0.35",
        f"hydra.run.dir={out_dir}",
    ]
    proc = subprocess.run(cmd, capture_output=True, text=True)
    log_text = proc.stdout + "\n" + proc.stderr
    if proc.returncode != 0:
        print(log_text[-1500:])
        continue
    prec_m = re.search(r"Precision=\s*([0-9.]+)", log_text)
    score_m = re.search(r"Matching_score=\s*([0-9.]+)", log_text)
    prec = prec_m.group(1) if prec_m else "N/A"
    score = score_m.group(1) if score_m else "N/A"
    results.append((label, prec, score, out_dir))
    print(f"Precision = {prec} | Matching_score = {score} | 输出目录: {out_dir}")

print("\n==================== 汇总对照表 ====================")
print(f"{'模型版本':35s} | {'Precision':10s} | {'Matching_score':14s}")
print("-" * 66)
for label, prec, score, _ in results:
    print(f"{label:35s} | {prec:10s} | {score:14s}")

### 步骤 9（GPU 运行时）: 可视化匹配结果（绿线=正确匹配，红线=误匹配）

In [ ]:
import glob, os
import matplotlib.pyplot as plt
import cv2

my_imgs = sorted(glob.glob(os.path.join(DRIVE_EVAL_DIR, "my_ltformer", "**", "matches*.png"), recursive=True))
if not my_imgs:
    print("未找到可视化结果图，请先运行步骤 8。")
else:
    show_imgs = my_imgs[:3]
    fig, axes = plt.subplots(len(show_imgs), 1, figsize=(14, 5 * len(show_imgs)))
    if len(show_imgs) == 1:
        axes = [axes]
    for ax, img_path in zip(axes, show_imgs):
        img = cv2.cvtColor(cv2.imread(img_path), cv2.COLOR_BGR2RGB)
        ax.imshow(img)
        ax.set_title(os.path.basename(img_path))
        ax.axis("off")
    plt.tight_layout()
    plt.show()